# 06 · Síntesis y verificación

Cada afirmación que puede ir al artículo se recalcula aquí desde las tablas procesadas y se comprueba con `assert`. Si un dato del SIAP cambia en una revisión, este notebook falla antes de que la cifra llegue al texto. El resultado queda en `outputs/tablas/insights_verificados.csv`.

In [1]:
import sys
from pathlib import Path

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))

import pandas as pd
import plotly.graph_objects as go

from src import config, tema

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)

estatal = pd.read_csv(config.DIR_PROCESSED / "cempasuchil_estatal.csv")
municipal = pd.read_csv(config.DIR_PROCESSED / "cempasuchil_municipal.csv")
FUENTE_SIAP = "Fuente: SIAP, Cierre de la Producción Agrícola 1980-2025. Análisis: Sahuaris."

In [2]:
insights = []

def registrar(id_, afirmacion, valor, fuente, notebook):
    insights.append({"id": id_, "afirmacion": afirmacion, "valor": valor, "fuente": fuente, "notebook": notebook})
    print(f"[{id_}] {afirmacion}")

doc = pd.read_csv(config.DIR_MANUAL / "cifras_documentales.csv").set_index("id")
ha_region = pd.read_csv(config.DIR_PROCESSED / "hectareas_por_region.csv", index_col="anio")
por_estado = pd.read_csv(config.DIR_PROCESSED / "hectareas_por_estado_2021_2025.csv", index_col="nomestado")
sonora = pd.read_csv(config.DIR_PROCESSED / "sonora_anual.csv", index_col="anio")
viaje = pd.read_csv(config.DIR_PROCESSED / "viaje_a_hermosillo.csv")
plantas = pd.read_csv(config.DIR_PROCESSED / "plantas_maceta.csv", index_col="anio")

## Lo nacional

In [3]:
pct_puebla = por_estado.loc["Puebla", "pct"]
assert 55 < pct_puebla < 70
registrar("puebla_pct", f"Puebla concentra {pct_puebla:.0f}% de la superficie sembrada de cempasúchil (promedio 2021-2025)", pct_puebla, "SIAP", "01")

ha_reciente = ha_region.loc[2021:2025, "total"].mean()
registrar("ha_reciente", f"México siembra unas {ha_reciente:,.0f} hectáreas de cempasúchil al año (2021-2025)", round(ha_reciente), "SIAP", "01")

pico = int(ha_region["total"].idxmax())
veces = ha_region["total"].max() / ha_reciente
assert pico in (1985, 1988) and veces > 7
registrar("pico_ha", f"En {pico} se sembraron {ha_region['total'].max():,.0f} ha, {veces:.0f} veces más que hoy", round(ha_region["total"].max()), "SIAP", "02")

assert ha_region.loc[1985, "total"] > doc.loc["ha_1985_gaceta", "valor"]
registrar("check_1985", "La suma del SIAP para 1985 confirma la cifra de Gaceta UNAM (más de 16,600 ha)", round(ha_region.loc[1985, "total"]), "SIAP + Gaceta UNAM", "02")

norte_80 = ha_region.loc[1980:1989, ["Sinaloa y Sonora", "Bajío (Gto., Mich., Qro.)"]].sum().sum() / ha_region.loc[1980:1989, "total"].sum() * 100
norte_hoy = ha_region.loc[2021:2025, ["Sinaloa y Sonora", "Bajío (Gto., Mich., Qro.)"]].sum().sum() / ha_region.loc[2021:2025, "total"].sum() * 100
assert norte_80 > 60 and norte_hoy < 5
forrajero = estatal[estatal["presentacion"] == "Forrajero (t)"]
ultimo_forrajero = int(forrajero["anio"].max())
assert ultimo_forrajero == 2001
registrar("ultimo_forrajero", f"El último cempasúchil forrajero (para pigmento) que registra el SIAP es de {ultimo_forrajero}, todo en Sinaloa", ultimo_forrajero, "SIAP", "02")

registrar("geografia", f"En los 80, Sinaloa, Sonora y el Bajío tenían {norte_80:.0f}% de la superficie; entre 2021 y 2025, {norte_hoy:.1f}%", round(norte_80, 1), "SIAP", "02")

[puebla_pct] Puebla concentra 66% de la superficie sembrada de cempasúchil (promedio 2021-2025)
[ha_reciente] México siembra unas 2,227 hectáreas de cempasúchil al año (2021-2025)
[pico_ha] En 1985 se sembraron 17,577 ha, 8 veces más que hoy
[check_1985] La suma del SIAP para 1985 confirma la cifra de Gaceta UNAM (más de 16,600 ha)
[ultimo_forrajero] El último cempasúchil forrajero (para pigmento) que registra el SIAP es de 2001, todo en Sinaloa
[geografia] En los 80, Sinaloa, Sonora y el Bajío tenían 83% de la superficie; entre 2021 y 2025, 2.5%


## Sonora

In [4]:
s_pico = int(sonora["volumenproduccion"].idxmax())
assert s_pico == 1982
registrar("sonora_pico", f"Sonora cosechó {sonora.loc[s_pico, 'volumenproduccion']:,.0f} t de cempasúchil en {s_pico} ({sonora.loc[s_pico, 'pct_toneladas_nacional']:.0f}% del país)", sonora.loc[s_pico, "volumenproduccion"], "SIAP", "03")

reciente = sonora.loc[2018:2024, "volumenproduccion"]
registrar("sonora_hoy", f"Entre 2018 y 2024 Sonora registró de {reciente.min():.0f} a {reciente.max():.0f} t al año, solo en Ímuris y Magdalena", reciente.max(), "SIAP", "03")
assert 2025 not in sonora.index
registrar("sonora_2025", "En el cierre 2025 el SIAP no registra cempasúchil en Sonora", 0, "SIAP", "03")

caida = sonora.loc[s_pico, "volumenproduccion"] / sonora.loc[2024, "volumenproduccion"]
registrar("sonora_caida", f"De {s_pico} a 2024 la cosecha sonorense se redujo {caida:,.0f} veces", round(caida), "SIAP", "03")

[sonora_pico] Sonora cosechó 17,525 t de cempasúchil en 1982 (15% del país)
[sonora_hoy] Entre 2018 y 2024 Sonora registró de 17 a 100 t al año, solo en Ímuris y Magdalena
[sonora_2025] En el cierre 2025 el SIAP no registra cempasúchil en Sonora
[sonora_caida] De 1982 a 2024 la cosecha sonorense se redujo 779 veces


## El viaje

In [5]:
media = (viaje["distancia_km"] * viaje["ha_promedio"]).sum() / viaje["ha_promedio"].sum()
assert 1500 < media < 2500
registrar("distancia_media", f"La flor sembrada en México está, en promedio, a {media:,.0f} km por carretera de Hermosillo", round(media), "SIAP + OSRM", "04")
cerca = viaje.loc[viaje["distancia_km"] < 500, "ha_promedio"].sum() / viaje["ha_promedio"].sum() * 100
assert cerca < 1
registrar("cerca_pct", f"Solo {cerca:.2f}% de la superficie está a menos de 500 km de Hermosillo", round(cerca, 2), "SIAP + OSRM", "04")

[distancia_media] La flor sembrada en México está, en promedio, a 1,990 km por carretera de Hermosillo
[cerca_pct] Solo 0.24% de la superficie está a menos de 500 km de Hermosillo


## La maceta y el mundo (documental)

In [6]:
p05, p25 = plantas.loc[2005, "total"], plantas.loc[2025, "total"]
registrar("plantas_crecen", f"Las plantas en maceta registradas por el SIAP pasaron de {p05 / 1e6:.2f} a {p25 / 1e6:.2f} millones entre 2005 y 2025", p25, "SIAP", "05")
registrar("cdmx_2026", f"La CDMX reporta {doc.loc['cdmx_plantas_2026', 'valor'] / 1e6:.1f} millones de plantas en 2026", doc.loc["cdmx_plantas_2026", "valor"], doc.loc["cdmx_plantas_2026", "fuente"], "05")
registrar("semilla", f"Se importa el {doc.loc['semilla_maceta_importada_pct', 'valor']}% de la semilla del cempasúchil en maceta, sobre todo de Estados Unidos (Consejo Mexicano de la Flor)", doc.loc["semilla_maceta_importada_pct", "valor"], doc.loc["semilla_maceta_importada_pct", "fuente"], "05")
registrar("china_india", f"China e India concentran cerca del {doc.loc['china_india_pct_industrial', 'valor']}% de la producción industrial de cempasúchil", doc.loc["china_india_pct_industrial", "valor"], doc.loc["china_india_pct_industrial", "fuente"], "—")
registrar("pigmento", "Desde 2000 México no produce pigmento industrial de cempasúchil; importa unos 100 millones de gramos al año, 90% de China", doc.loc["pigmento_importado_g", "valor"], doc.loc["pigmento_importado_g", "fuente"], "—")
china_ha = doc.loc["china_ha_empresa", "valor"]
registrar("china_empresa", f"Una sola empresa china siembra {china_ha:,} ha, {china_ha / ha_reciente:.1f} veces toda la superficie mexicana actual", china_ha, doc.loc["china_ha_empresa", "fuente"], "—")

[plantas_crecen] Las plantas en maceta registradas por el SIAP pasaron de 0.36 a 3.02 millones entre 2005 y 2025
[cdmx_2026] La CDMX reporta 9.9 millones de plantas en 2026
[semilla] Se importa el 99% de la semilla del cempasúchil en maceta, sobre todo de Estados Unidos (Consejo Mexicano de la Flor)
[china_india] China e India concentran cerca del 95% de la producción industrial de cempasúchil
[pigmento] Desde 2000 México no produce pigmento industrial de cempasúchil; importa unos 100 millones de gramos al año, 90% de China
[china_empresa] Una sola empresa china siembra 6,000 ha, 2.7 veces toda la superficie mexicana actual


In [7]:
config.DIR_TABLAS.mkdir(parents=True, exist_ok=True)
tabla = pd.DataFrame(insights)
tabla.to_csv(config.DIR_TABLAS / "insights_verificados.csv", index=False)
print(f"{len(tabla)} afirmaciones verificadas")
tabla[["id", "afirmacion"]]

18 afirmaciones verificadas


,id,afirmacion
0,puebla_pct,Puebla concentra 66% de la superficie sembrada...
1,ha_reciente,"México siembra unas 2,227 hectáreas de cempasú..."
2,pico_ha,"En 1985 se sembraron 17,577 ha, 8 veces más qu..."
3,check_1985,La suma del SIAP para 1985 confirma la cifra d...
4,ultimo_forrajero,El último cempasúchil forrajero (para pigmento...
5,geografia,"En los 80, Sinaloa, Sonora y el Bajío tenían 8..."
6,sonora_pico,"Sonora cosechó 17,525 t de cempasúchil en 1982..."
7,sonora_hoy,Entre 2018 y 2024 Sonora registró de 17 a 100 ...
8,sonora_2025,En el cierre 2025 el SIAP no registra cempasúc...
9,sonora_caida,De 1982 a 2024 la cosecha sonorense se redujo ...
